# Point Cloud Volume Estimation — Research Pipeline

A complete, multi-model deep-learning pipeline for **regression on 3-D LAS point clouds**.

Change `MODEL_NAME` in **Section 2** and run all cells end-to-end.

## 1. Imports

In [ ]:
# ─── Standard Library ──────────────────────────────────────────────────────────
import os
import csv
import json
import math
import time
import random
import warnings
import logging
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Any

# ─── Scientific Stack ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ─── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ─── Deep Learning ─────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.cuda.amp import autocast, GradScaler

# ─── Point Cloud I/O ───────────────────────────────────────────────────────────
import laspy

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)s  %(message)s")
log = logging.getLogger(__name__)

## 2. Configuration

> **Change `MODEL_NAME` here to switch models and re-run the notebook.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ★  SINGLE CONTROL VARIABLE  ★                                              ║
# ║  Options: "PointNet" | "PointNetPP" | "PointNext" | "PointMAE" | "DGCNN"  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
MODEL_NAME: str = "PointNet"

# ─── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR:        str = "sample_data"
LOGS_DIR:        str = "logs"
CHECKPOINTS_DIR: str = "checkpoints"
RESULTS_DIR:     str = "results"
PLOTS_DIR:       str = "plots"

for _d in [LOGS_DIR, CHECKPOINTS_DIR, RESULTS_DIR, PLOTS_DIR]:
    os.makedirs(_d, exist_ok=True)

# ─── Data ──────────────────────────────────────────────────────────────────────
CSV_FILE:    str = os.path.join(DATA_DIR, "volume.csv")
NUM_POINTS:  int = 2048       # Points per cloud after FPS / random re-sample
TEST_SIZE:   float = 0.15     # 70 / 15 / 15 split
VAL_SIZE:    float = 0.15
RANDOM_SEED: int = 42

# ─── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE:        int   = 16
NUM_EPOCHS:        int   = 100
LEARNING_RATE:     float = 1e-3
WEIGHT_DECAY:      float = 1e-4
GRAD_CLIP:         float = 1.0
T0:                int   = 10    # CosineAnnealingWarmRestarts period
T_MULT:            int   = 2
EARLY_STOPPING_PATIENCE: int = 20   # epochs without val improvement
NUM_WORKERS:       int   = 4        # DataLoader workers
PIN_MEMORY:        bool  = True

# ─── Model-specific knobs ──────────────────────────────────────────────────────
DGCNN_K:           int = 20      # k-nearest neighbours for DGCNN
POINTNETPP_RADII:  List[float] = [0.2, 0.4]
POINTMAE_MASK_RATIO: float = 0.75

# ─── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()   # Mixed precision only on CUDA
print(f"Device  : {DEVICE}")
print(f"AMP     : {USE_AMP}")
print(f"Workers : {NUM_WORKERS}")

## 3. Reproducibility

In [ ]:
def set_seed(seed: int = RANDOM_SEED) -> None:
    """Fix all random seeds for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)
print(f"Seed set to {RANDOM_SEED}")

## 4. Data Preprocessing

Scan the LAS files and build a summary DataFrame (re-uses original notebook logic).

In [ ]:
def scan_las_directory(src: str) -> pd.DataFrame:
    """
    Walk *src* and count points in every .las file.
    Returns a DataFrame with columns [filename, num_points].
    Handles missing / corrupted files gracefully.
    """
    records = []
    for fp in sorted(Path(src).glob("*.las")):
        try:
            las = laspy.read(str(fp))
            n = len(las.x)
            records.append({"filename": fp.name, "num_points": n})
        except Exception as exc:
            log.warning(f"Skipping {fp.name}: {exc}")
    if not records:
        log.warning("No .las files found in %s", src)
    df = pd.DataFrame(records)
    return df

if os.path.isdir(DATA_DIR):
    scan_df = scan_las_directory(DATA_DIR)
    if not scan_df.empty:
        print(f"Found {len(scan_df)} LAS files")
        display(scan_df.describe())
    else:
        print("No LAS files found — will still build the Dataset from CSV.")
else:
    print(f"DATA_DIR '{DATA_DIR}' not found — create it before training.")
    scan_df = pd.DataFrame()

## 5. Dataset Class

In [ ]:
class PointCloudDataset(Dataset):
    """
    Loads LAS point clouds and returns (points_tensor, volume_tensor).

    * Applies Farthest Point Sampling (FPS) for N > num_points.
    * Random duplicate-sample for N < num_points.
    * Zero-mean centering + unit-sphere normalisation.
    * Optional augmentation (Z-axis rotation + Gaussian jitter).
    """

    def __init__(
        self,
        df: pd.DataFrame,
        data_dir: str,
        num_points: int = NUM_POINTS,
        train: bool = True,
    ) -> None:
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.num_points = num_points
        self.train = train

    def __len__(self) -> int:
        return len(self.df)

    # ── Sampling ────────────────────────────────────────────────────────────────
    @staticmethod
    def _fps(xyz: np.ndarray, npoint: int) -> np.ndarray:
        """Farthest-point sampling (O(N·k)). Reused from original notebook."""
        N = xyz.shape[0]
        centroids = np.zeros(npoint, dtype=np.int32)
        dist = np.ones(N) * 1e10
        farthest = np.random.randint(0, N)
        for i in range(npoint):
            centroids[i] = farthest
            cx, cy, cz = xyz[farthest]
            d = (xyz[:, 0] - cx) ** 2 + (xyz[:, 1] - cy) ** 2 + (xyz[:, 2] - cz) ** 2
            mask = d < dist
            dist[mask] = d[mask]
            farthest = int(np.argmax(dist))
        return xyz[centroids]

    def _sample(self, xyz: np.ndarray) -> np.ndarray:
        N = xyz.shape[0]
        if N == 0:
            raise ValueError("Empty point cloud after loading.")
        if N > self.num_points:
            return self._fps(xyz, self.num_points)
        idx = np.random.choice(N, self.num_points, replace=True)
        return xyz[idx]

    # ── Normalisation ───────────────────────────────────────────────────────────
    @staticmethod
    def _normalise(pts: np.ndarray) -> np.ndarray:
        """Zero-mean + unit sphere (matches standard PointNet pre-processing)."""
        pts = pts - pts.mean(axis=0)
        scale = np.max(np.sqrt(np.sum(pts ** 2, axis=1)))
        if scale > 0:
            pts = pts / scale
        return pts

    # ── Augmentation ────────────────────────────────────────────────────────────
    @staticmethod
    def _augment(pts: np.ndarray) -> np.ndarray:
        """Z-axis random rotation + per-point Gaussian jitter."""
        theta = np.random.uniform(0, 2 * np.pi)
        c, s = np.cos(theta), np.sin(theta)
        R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float32)
        pts = pts @ R.T
        pts += np.random.normal(0, 0.01, size=pts.shape).astype(np.float32)
        return pts

    # ── __getitem__ ─────────────────────────────────────────────────────────────
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        row = self.df.iloc[idx]
        las_path = os.path.join(self.data_dir, row.iloc[0])
        gt_vol = float(row.iloc[1])

        # --- load ---
        try:
            las = laspy.read(las_path)
            pts = np.vstack((las.x, las.y, las.z)).T.astype(np.float32)
        except Exception as exc:
            log.warning(f"Cannot read {las_path}: {exc}. Returning zeros.")
            pts = np.zeros((self.num_points, 3), dtype=np.float32)

        # --- sanitise ---
        pts = pts[np.isfinite(pts).all(axis=1)]   # drop NaN/Inf rows
        if len(pts) == 0:
            pts = np.zeros((self.num_points, 3), dtype=np.float32)

        pts = self._sample(pts)
        pts = self._normalise(pts)

        if self.train:
            pts = self._augment(pts)

        # shape: (3, N) — channels-first for Conv1d
        t_pts = torch.from_numpy(pts.T).float()
        t_vol = torch.tensor(gt_vol, dtype=torch.float32)
        return t_pts, t_vol

## 6. Train / Validation / Test Split

In [ ]:
def load_and_split(
    csv_path: str,
    test_size: float = TEST_SIZE,
    val_size: float = VAL_SIZE,
    seed: int = RANDOM_SEED,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Reads the master CSV and returns (train_df, val_df, test_df).
    Expects columns: [filename, volume].
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    df = pd.read_csv(csv_path)
    log.info(f"Loaded CSV  →  {len(df)} rows   columns: {list(df.columns)}")

    # drop rows where the .las file is missing
    valid_mask = df.iloc[:, 0].apply(
        lambda fn: os.path.exists(os.path.join(DATA_DIR, fn))
    )
    if (~valid_mask).any():
        log.warning(f"Dropping {(~valid_mask).sum()} rows with missing LAS files.")
    df = df[valid_mask].reset_index(drop=True)

    # stratified split is not applicable to regression, so plain random split
    train_df, tmp_df = train_test_split(df, test_size=(test_size + val_size), random_state=seed)
    relative_val = val_size / (test_size + val_size)
    val_df, test_df = train_test_split(tmp_df, test_size=relative_val, random_state=seed)

    print(f"Split  →  train: {len(train_df)}   val: {len(val_df)}   test: {len(test_df)}")
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )

train_df, val_df, test_df = load_and_split(CSV_FILE)

# persist splits so they can be reloaded without re-splitting
train_df.to_csv(os.path.join(DATA_DIR, "split_train.csv"), index=False)
val_df.to_csv(os.path.join(DATA_DIR,   "split_val.csv"),   index=False)
test_df.to_csv(os.path.join(DATA_DIR,  "split_test.csv"),  index=False)

## 7. DataLoaders

In [ ]:
def make_loaders(
    train_df: pd.DataFrame,
    val_df:   pd.DataFrame,
    test_df:  pd.DataFrame,
    data_dir: str  = DATA_DIR,
    num_pts:  int  = NUM_POINTS,
    batch:    int  = BATCH_SIZE,
    workers:  int  = NUM_WORKERS,
    pin:      bool = PIN_MEMORY,
) -> Tuple[DataLoader, DataLoader, DataLoader]:

    train_ds = PointCloudDataset(train_df, data_dir, num_pts, train=True)
    val_ds   = PointCloudDataset(val_df,   data_dir, num_pts, train=False)
    test_ds  = PointCloudDataset(test_df,  data_dir, num_pts, train=False)

    kw = dict(num_workers=workers, pin_memory=pin)
    tr_loader  = DataLoader(train_ds, batch_size=batch, shuffle=True,  drop_last=True,  **kw)
    val_loader = DataLoader(val_ds,   batch_size=batch, shuffle=False, drop_last=False, **kw)
    tst_loader = DataLoader(test_ds,  batch_size=batch, shuffle=False, drop_last=False, **kw)

    print(f"Loaders  →  train: {len(tr_loader)} batches   "
          f"val: {len(val_loader)}   test: {len(tst_loader)}")
    return tr_loader, val_loader, tst_loader

train_loader, val_loader, test_loader = make_loaders(train_df, val_df, test_df)

# quick shape check
pts_sample, vol_sample = next(iter(train_loader))
print(f"Batch shapes  →  points: {tuple(pts_sample.shape)}   volumes: {tuple(vol_sample.shape)}")

## 8. Metrics & Common Utilities

In [ ]:
# ─── Metric helpers ────────────────────────────────────────────────────────────

def mape_safe(pred: np.ndarray, target: np.ndarray, eps: float = 1e-8) -> float:
    """Mean Absolute Percentage Error — safe against zero targets."""
    return float(np.mean(np.abs((pred - target) / (np.abs(target) + eps))) * 100)

def compute_metrics(pred: np.ndarray, target: np.ndarray) -> Dict[str, float]:
    """Return dict with MAE, RMSE, MSE, R2, MAPE."""
    mse  = float(mean_squared_error(target, pred))
    rmse = float(np.sqrt(mse))
    mae  = float(mean_absolute_error(target, pred))
    r2   = float(r2_score(target, pred))
    mape = mape_safe(pred, target)
    return dict(MAE=mae, RMSE=rmse, MSE=mse, R2=r2, MAPE=mape)

def param_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def torch_metrics(pred: torch.Tensor, target: torch.Tensor) -> Dict[str, float]:
    """Compute metrics directly on GPU tensors (no CPU copy mid-loop)."""
    p = pred.detach().float()
    t = target.detach().float()
    mse  = F.mse_loss(p, t).item()
    mae  = F.l1_loss(p, t).item()
    rmse = math.sqrt(mse)
    eps  = 1e-8
    mape = ((p - t).abs() / (t.abs() + eps)).mean().item() * 100
    # R² requires CPU numpy
    pn = p.cpu().numpy()
    tn = t.cpu().numpy()
    r2 = float(r2_score(tn, pn))
    return dict(MAE=mae, RMSE=rmse, MSE=mse, R2=r2, MAPE=mape)

# ─── CSV logger ────────────────────────────────────────────────────────────────

EPOCH_CSV_COLS = [
    "Epoch", "LR",
    "Train_Loss", "Val_Loss",
    "Train_MAE",  "Val_MAE",
    "Train_RMSE", "Val_RMSE",
    "Train_MSE",  "Val_MSE",
    "Train_R2",   "Val_R2",
    "Train_MAPE", "Val_MAPE",
    "Epoch_Time_s",
]

class EpochCSVLogger:
    """Append one row per epoch to a CSV (creates header on first call)."""

    def __init__(self, path: str) -> None:
        self.path = path
        self._init = os.path.exists(path)

    def log(self, row: Dict[str, Any]) -> None:
        new = not os.path.exists(self.path)
        with open(self.path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=EPOCH_CSV_COLS)
            if new:
                writer.writeheader()
            writer.writerow({k: row.get(k, "") for k in EPOCH_CSV_COLS})

print("Metrics & utilities ready.")

## 9. Base Training Engine

In [ ]:
class Trainer:
    """
    Reusable training engine.

    Features
    ────────
    • Mixed-precision AMP (CUDA only)
    • Gradient clipping
    • CosineAnnealingWarmRestarts scheduler
    • Early stopping
    • Best-checkpoint saving
    • Per-epoch CSV logging
    • Resume from checkpoint
    """

    def __init__(
        self,
        model:        nn.Module,
        model_name:   str,
        train_loader: DataLoader,
        val_loader:   DataLoader,
        device:       torch.device      = DEVICE,
        lr:           float             = LEARNING_RATE,
        weight_decay: float             = WEIGHT_DECAY,
        grad_clip:    float             = GRAD_CLIP,
        num_epochs:   int               = NUM_EPOCHS,
        patience:     int               = EARLY_STOPPING_PATIENCE,
        use_amp:      bool              = USE_AMP,
        ckpt_dir:     str               = CHECKPOINTS_DIR,
        logs_dir:     str               = LOGS_DIR,
    ) -> None:
        self.model        = model.to(device)
        self.model_name   = model_name
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.device       = device
        self.num_epochs   = num_epochs
        self.patience     = patience
        self.grad_clip    = grad_clip
        self.use_amp      = use_amp and torch.cuda.is_available()
        self.ckpt_path    = os.path.join(ckpt_dir, f"{model_name.lower()}_best.pth")
        csv_path          = os.path.join(logs_dir, f"{model_name.lower()}_train.csv")
        self.csv_logger   = EpochCSVLogger(csv_path)

        # Multi-GPU wrapper
        if torch.cuda.device_count() > 1:
            self.model = nn.DataParallel(self.model)
            log.info(f"Using {torch.cuda.device_count()} GPUs.")

        self.criterion = nn.MSELoss()
        self.optimizer = AdamW(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        self.scheduler = CosineAnnealingWarmRestarts(self.optimizer, T_0=T0, T_mult=T_MULT)
        self.scaler    = GradScaler(enabled=self.use_amp)

        self.best_val_loss    = float("inf")
        self.no_improve_count = 0
        self.total_train_time = 0.0
        self.history: List[Dict] = []

    # ── Inner loops ─────────────────────────────────────────────────────────────

    def _run_epoch(self, loader: DataLoader, train: bool) -> Tuple[float, Dict]:
        self.model.train(train)
        total_loss = 0.0
        all_pred, all_tgt = [], []

        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for pts, vols in loader:
                pts  = pts.to(self.device, non_blocking=True)
                vols = vols.to(self.device, non_blocking=True)

                with autocast(enabled=self.use_amp):
                    pred = self.model(pts)
                    if pred.dim() == 0:
                        pred = pred.unsqueeze(0)
                    loss = self.criterion(pred, vols)

                if train:
                    self.optimizer.zero_grad(set_to_none=True)
                    self.scaler.scale(loss).backward()
                    self.scaler.unscale_(self.optimizer)
                    nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()

                bs = pts.size(0)
                total_loss += loss.item() * bs
                all_pred.append(pred.detach().cpu().float())
                all_tgt.append(vols.detach().cpu().float())

        n = len(loader.dataset)
        avg_loss = total_loss / n
        pn = torch.cat(all_pred).numpy()
        tn = torch.cat(all_tgt).numpy()
        metrics = compute_metrics(pn, tn)
        return avg_loss, metrics

    # ── Public interface ─────────────────────────────────────────────────────────

    def train(self, resume: bool = False) -> Dict:
        """
        Run the full training loop.
        Returns the best validation metrics dict.
        """
        if resume and os.path.exists(self.ckpt_path):
            ckpt = torch.load(self.ckpt_path, map_location=self.device)
            mdl  = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
            mdl.load_state_dict(ckpt["model_state"])
            self.optimizer.load_state_dict(ckpt["optimizer_state"])
            start_epoch = ckpt.get("epoch", 0) + 1
            self.best_val_loss = ckpt.get("best_val_loss", float("inf"))
            log.info(f"Resumed from epoch {start_epoch}.")
        else:
            start_epoch = 0

        best_metrics: Dict = {}

        for epoch in range(start_epoch, self.num_epochs):
            t0 = time.time()

            tr_loss, tr_metrics = self._run_epoch(self.train_loader, train=True)
            vl_loss, vl_metrics = self._run_epoch(self.val_loader,   train=False)

            self.scheduler.step(epoch + 1)   # epoch-indexed scheduler step
            epoch_t = time.time() - t0
            self.total_train_time += epoch_t

            lr_now = self.optimizer.param_groups[0]["lr"]

            # ── CSV row ──────────────────────────────────────────────────────────
            row = {
                "Epoch":       epoch + 1,
                "LR":          round(lr_now, 8),
                "Train_Loss":  round(tr_loss, 6),     "Val_Loss":   round(vl_loss, 6),
                "Train_MAE":   round(tr_metrics["MAE"], 6), "Val_MAE":  round(vl_metrics["MAE"], 6),
                "Train_RMSE":  round(tr_metrics["RMSE"],6), "Val_RMSE": round(vl_metrics["RMSE"],6),
                "Train_MSE":   round(tr_metrics["MSE"], 6), "Val_MSE":  round(vl_metrics["MSE"], 6),
                "Train_R2":    round(tr_metrics["R2"],  6), "Val_R2":   round(vl_metrics["R2"],  6),
                "Train_MAPE":  round(tr_metrics["MAPE"],4), "Val_MAPE": round(vl_metrics["MAPE"],4),
                "Epoch_Time_s":round(epoch_t, 2),
            }
            self.csv_logger.log(row)
            self.history.append(row)

            # ── checkpoint ───────────────────────────────────────────────────────
            save_flag = ""
            if vl_loss < self.best_val_loss:
                self.best_val_loss    = vl_loss
                self.no_improve_count = 0
                best_metrics          = vl_metrics.copy()
                mdl = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
                torch.save({
                    "epoch":          epoch,
                    "model_state":    mdl.state_dict(),
                    "optimizer_state":self.optimizer.state_dict(),
                    "best_val_loss":  self.best_val_loss,
                }, self.ckpt_path)
                save_flag = "  💾"
            else:
                self.no_improve_count += 1

            # ── progress ─────────────────────────────────────────────────────────
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(
                    f"[{epoch+1:4d}/{self.num_epochs}]  "
                    f"tr_loss={tr_loss:.4f}  vl_loss={vl_loss:.4f}  "
                    f"vl_MAE={vl_metrics['MAE']:.4f}  vl_R2={vl_metrics['R2']:.4f}  "
                    f"lr={lr_now:.2e}{save_flag}"
                )

            # ── early stopping ────────────────────────────────────────────────────
            if self.no_improve_count >= self.patience:
                print(f"\nEarly stopping triggered at epoch {epoch+1} "
                      f"(no improvement for {self.patience} epochs).")
                break

        print(f"\nTotal training time: {self.total_train_time/60:.2f} min")
        return best_metrics

## 10. Base Testing Engine

In [ ]:
class Evaluator:
    """
    Load the best checkpoint and evaluate on the held-out test set.
    Produces all required plots and returns a metrics dict.
    """

    def __init__(
        self,
        model:       nn.Module,
        model_name:  str,
        test_loader: DataLoader,
        device:      torch.device = DEVICE,
        ckpt_dir:    str          = CHECKPOINTS_DIR,
        plots_dir:   str          = PLOTS_DIR,
    ) -> None:
        self.model       = model.to(device)
        self.model_name  = model_name
        self.test_loader = test_loader
        self.device      = device
        self.ckpt_path   = os.path.join(ckpt_dir, f"{model_name.lower()}_best.pth")
        self.plots_dir   = plots_dir

    def load_best(self) -> None:
        if not os.path.exists(self.ckpt_path):
            raise FileNotFoundError(f"Checkpoint not found: {self.ckpt_path}")
        ckpt = torch.load(self.ckpt_path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state"])
        log.info(f"Loaded best checkpoint from {self.ckpt_path}")

    def predict(self) -> Tuple[np.ndarray, np.ndarray]:
        self.model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for pts, vols in self.test_loader:
                pts = pts.to(self.device, non_blocking=True)
                out = self.model(pts)
                if out.dim() == 0:
                    out = out.unsqueeze(0)
                preds.append(out.cpu().numpy())
                targets.append(vols.numpy())
        return np.concatenate(preds), np.concatenate(targets)

    def evaluate(self) -> Dict[str, float]:
        self.load_best()
        t0     = time.time()
        pred, tgt = self.predict()
        inf_t  = time.time() - t0

        metrics = compute_metrics(pred, tgt)
        metrics["Inference_Time_s"] = round(inf_t, 4)

        print("\n" + "=" * 60)
        print(f"  {self.model_name}  —  Test-set Evaluation")
        print("=" * 60)
        for k, v in metrics.items():
            print(f"  {k:20s}: {v:.6f}")
        print("=" * 60)

        self._plot_scatter(pred, tgt)
        self._plot_residual(pred, tgt)
        self._plot_error_hist(pred, tgt)

        return metrics

    # ── Plots ────────────────────────────────────────────────────────────────────

    def _save(self, fig: plt.Figure, name: str) -> None:
        path = os.path.join(self.plots_dir, f"{self.model_name.lower()}_{name}.png")
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"  Saved → {path}")

    def _plot_scatter(self, pred: np.ndarray, tgt: np.ndarray) -> None:
        fig, ax = plt.subplots(figsize=(5, 5))
        lo, hi  = min(tgt.min(), pred.min()), max(tgt.max(), pred.max())
        ax.scatter(tgt, pred, alpha=0.6, s=20, edgecolors="none")
        ax.plot([lo, hi], [lo, hi], "r--", lw=1.5, label="Perfect")
        ax.set_xlabel("Ground Truth Volume")
        ax.set_ylabel("Predicted Volume")
        ax.set_title(f"{self.model_name}  –  Predicted vs Ground Truth")
        ax.legend()
        self._save(fig, "scatter")

    def _plot_residual(self, pred: np.ndarray, tgt: np.ndarray) -> None:
        residual = pred - tgt
        fig, ax  = plt.subplots(figsize=(6, 4))
        ax.scatter(tgt, residual, alpha=0.6, s=20, edgecolors="none")
        ax.axhline(0, color="r", lw=1.5, linestyle="--")
        ax.set_xlabel("Ground Truth Volume")
        ax.set_ylabel("Residual (Pred − True)")
        ax.set_title(f"{self.model_name}  –  Residual Plot")
        self._save(fig, "residual")

    def _plot_error_hist(self, pred: np.ndarray, tgt: np.ndarray) -> None:
        err = pred - tgt
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(err, bins=30, edgecolor="black", alpha=0.8)
        ax.axvline(0,          color="r", lw=1.5, linestyle="--", label="Zero")
        ax.axvline(err.mean(), color="g", lw=1.5, linestyle="--", label=f"Mean={err.mean():.3f}")
        ax.set_xlabel("Prediction Error")
        ax.set_ylabel("Count")
        ax.set_title(f"{self.model_name}  –  Error Histogram")
        ax.legend()
        self._save(fig, "error_hist")

print("Evaluator class ready.")

## 11. Training History Plotter

In [ ]:
def plot_training_history(history: List[Dict], model_name: str, plots_dir: str = PLOTS_DIR) -> None:
    """
    Generate all required training-curve plots from the epoch-history list.
    """
    if not history:
        print("No history to plot.")
        return

    df = pd.DataFrame(history)

    PAIRS = [
        ("Train_Loss",  "Val_Loss",  "MSE Loss",   "loss_curve"),
        ("Train_MAE",   "Val_MAE",   "MAE",         "mae_curve"),
        ("Train_RMSE",  "Val_RMSE",  "RMSE",        "rmse_curve"),
        ("Train_R2",    "Val_R2",    "R² Score",    "r2_curve"),
        ("Train_MAPE",  "Val_MAPE",  "MAPE (%)",    "mape_curve"),
    ]

    for tr_col, vl_col, ylabel, tag in PAIRS:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(df["Epoch"], df[tr_col], label="Train")
        ax.plot(df["Epoch"], df[vl_col], label="Val")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{model_name}  –  {ylabel}")
        ax.legend()
        path = os.path.join(plots_dir, f"{model_name.lower()}_{tag}.png")
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"  Saved → {path}")

    # LR curve
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(df["Epoch"], df["LR"])
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Learning Rate")
    ax.set_title(f"{model_name}  –  LR Schedule")
    ax.set_yscale("log")
    path = os.path.join(plots_dir, f"{model_name.lower()}_lr_curve.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")

print("History plotter ready.")

## 12. Model Implementations

### 12.1 PointNet

In [ ]:
# ─── Input Transform (T-Net) ───────────────────────────────────────────────────
class TNet(nn.Module):
    """Spatial transformer network (3×3 or k×k matrix prediction)."""

    def __init__(self, k: int = 3) -> None:
        super().__init__()
        self.k = k
        self.conv1 = nn.Sequential(nn.Conv1d(k, 64, 1),  nn.BatchNorm1d(64),  nn.ReLU())
        self.conv2 = nn.Sequential(nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU())
        self.conv3 = nn.Sequential(nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU())
        self.fc1   = nn.Sequential(nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU())
        self.fc2   = nn.Sequential(nn.Linear(512,  256), nn.BatchNorm1d(256), nn.ReLU())
        self.fc3   = nn.Linear(256, k * k)
        nn.init.zeros_(self.fc3.weight)
        nn.init.zeros_(self.fc3.bias)
        self.identity = torch.eye(k).view(-1)   # for residual init

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.size(0)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.max(dim=-1)[0]          # global max pool
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        eye = self.identity.to(x.device).expand(B, -1)
        x   = x + eye
        return x.view(B, self.k, self.k)


class PointNet(nn.Module):
    """
    PointNet (Qi et al., 2017) — regression head.
    With input & feature transforms (T-Nets).
    """

    def __init__(self, dropout: float = 0.3) -> None:
        super().__init__()
        self.tnet3  = TNet(3)
        self.conv1  = nn.Sequential(nn.Conv1d(3, 64, 1),   nn.BatchNorm1d(64),   nn.ReLU())
        self.conv2  = nn.Sequential(nn.Conv1d(64, 64, 1),  nn.BatchNorm1d(64),   nn.ReLU())
        self.tnet64 = TNet(64)
        self.conv3  = nn.Sequential(nn.Conv1d(64, 64, 1),  nn.BatchNorm1d(64),   nn.ReLU())
        self.conv4  = nn.Sequential(nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128),  nn.ReLU())
        self.conv5  = nn.Sequential(nn.Conv1d(128, 1024, 1),nn.BatchNorm1d(1024), nn.ReLU())

        self.fc1    = nn.Sequential(nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout))
        self.fc2    = nn.Sequential(nn.Linear(512, 256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout))
        self.fc3    = nn.Linear(256, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, 3, N)"""
        # input transform
        t3 = self.tnet3(x)
        x  = torch.bmm(t3, x)

        x  = self.conv1(x)
        x  = self.conv2(x)

        # feature transform
        t64 = self.tnet64(x)
        x   = torch.bmm(t64, x)

        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)

        # global max pool
        x = x.max(dim=-1)[0]   # (B, 1024)

        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x).squeeze(-1)   # (B,)
        return x

print(f"PointNet params: {param_count(PointNet()):,}")

### 12.2 PointNet++

In [ ]:
# ─── PointNet++ helpers ────────────────────────────────────────────────────────

def _square_distance(src: torch.Tensor, dst: torch.Tensor) -> torch.Tensor:
    """(B, N, C) × (B, M, C) → (B, N, M) squared L2 distances."""
    return (
        torch.sum(src ** 2, -1, keepdim=True)
        - 2 * torch.bmm(src, dst.transpose(1, 2))
        + torch.sum(dst ** 2, -1, keepdim=True).transpose(1, 2)
    )

def _fps_torch(xyz: torch.Tensor, npoint: int) -> torch.Tensor:
    """GPU farthest-point sampling. xyz: (B, N, 3). Returns (B, npoint)."""
    B, N, _ = xyz.shape
    device   = xyz.device
    centroids    = torch.zeros(B, npoint, dtype=torch.long, device=device)
    distance     = torch.full((B, N), 1e10, device=device)
    farthest     = torch.randint(0, N, (B,), dtype=torch.long, device=device)
    batch_idx    = torch.arange(B, device=device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid         = xyz[batch_idx, farthest, :].unsqueeze(1)  # (B,1,3)
        dist             = ((xyz - centroid) ** 2).sum(-1)
        mask             = dist < distance
        distance[mask]   = dist[mask]
        farthest         = distance.argmax(-1)
    return centroids

def _ball_query(xyz: torch.Tensor, new_xyz: torch.Tensor, radius: float, nsample: int) -> torch.Tensor:
    """
    For each point in new_xyz, find up to nsample neighbours within radius in xyz.
    Returns idx (B, M, nsample).
    """
    B, N, _ = xyz.shape
    M       = new_xyz.shape[1]
    device  = xyz.device
    sqr_r   = radius ** 2

    sq_dist = _square_distance(new_xyz, xyz)              # (B, M, N)
    group_idx = sq_dist.argsort(dim=-1)[:, :, :nsample]  # (B, M, nsample)

    # replace out-of-radius with idx of the first valid (i.e. nearest)
    sq_first = sq_dist.gather(-1, group_idx[:, :, :1].expand_as(group_idx))
    out_mask = sq_dist.gather(-1, group_idx) > sqr_r
    group_idx[out_mask] = group_idx[:, :, :1].expand_as(group_idx)[out_mask]
    return group_idx

def _index_points(pts: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    """pts: (B,N,C)  idx: (B,...) → (B,...,C)"""
    B = pts.shape[0]
    view_shape = list(idx.shape) + [pts.shape[-1]]
    idx_exp    = idx.unsqueeze(-1).expand(*view_shape)
    pts_exp    = pts.unsqueeze(1).expand(B, *([1] * (len(idx.shape) - 1)), pts.shape[1], pts.shape[2])
    # simpler approach
    flat_idx   = idx.reshape(B, -1)
    flat_pts   = pts[torch.arange(B, device=pts.device).unsqueeze(1), flat_idx]  # (B,?,C)
    return flat_pts.reshape(*view_shape)


class SetAbstraction(nn.Module):
    """PointNet++ Set Abstraction with multi-radius ball query."""

    def __init__(self, npoint: int, radius: float, nsample: int, in_ch: int, mlp_chs: List[int]) -> None:
        super().__init__()
        self.npoint  = npoint
        self.radius  = radius
        self.nsample = nsample

        layers: List[nn.Module] = []
        last = in_ch + 3
        for out_ch in mlp_chs:
            layers += [nn.Conv2d(last, out_ch, 1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
            last = out_ch
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz: torch.Tensor, points: Optional[torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        xyz:    (B, N, 3)
        points: (B, N, C) or None
        returns new_xyz (B,M,3), new_pts (B,M,mlp[-1])
        """
        B, N, _ = xyz.shape
        fps_idx  = _fps_torch(xyz, self.npoint)               # (B, M)
        new_xyz  = _index_points(xyz, fps_idx)                # (B, M, 3)

        idx      = _ball_query(xyz, new_xyz, self.radius, self.nsample)  # (B,M,ns)
        grouped  = _index_points(xyz, idx.reshape(B, -1)).reshape(B, self.npoint, self.nsample, 3)
        grouped -= new_xyz.unsqueeze(2)

        if points is not None:
            grouped_pts = _index_points(points, idx.reshape(B, -1)).reshape(
                B, self.npoint, self.nsample, -1
            )
            grouped = torch.cat([grouped, grouped_pts], dim=-1)  # (B,M,ns,3+C)

        grouped = grouped.permute(0, 3, 2, 1)  # (B, C, ns, M)
        grouped = self.mlp(grouped)
        new_pts = grouped.max(dim=2)[0].permute(0, 2, 1)  # (B, M, out_ch)
        return new_xyz, new_pts


class GlobalSetAbstraction(nn.Module):
    """Group-all (no ball query) — final SA layer."""

    def __init__(self, in_ch: int, mlp_chs: List[int]) -> None:
        super().__init__()
        layers: List[nn.Module] = []
        last = in_ch + 3
        for out_ch in mlp_chs:
            layers += [nn.Conv1d(last, out_ch, 1), nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True)]
            last = out_ch
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz: torch.Tensor, points: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        combined = torch.cat([xyz, points], dim=-1).permute(0, 2, 1)  # (B, C+3, N)
        out = self.mlp(combined).max(dim=-1)[0]   # (B, mlp[-1])
        return None, out.unsqueeze(1)


class PointNetPP(nn.Module):
    """PointNet++ SSG (single-scale grouping) for regression."""

    def __init__(self, dropout: float = 0.4) -> None:
        super().__init__()
        self.sa1 = SetAbstraction(512, 0.2, 32, 0,   [64,  64,  128])
        self.sa2 = SetAbstraction(128, 0.4, 64, 128,  [128, 128, 256])
        self.sa3 = GlobalSetAbstraction(256, [256, 512, 1024])

        self.fc1     = nn.Sequential(nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout))
        self.fc2     = nn.Sequential(nn.Linear(512,  256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout))
        self.fc3     = nn.Linear(256, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, 3, N)"""
        xyz = x.permute(0, 2, 1)   # (B, N, 3)
        xyz, pts = self.sa1(xyz, None)
        xyz, pts = self.sa2(xyz, pts)
        _,   pts = self.sa3(xyz, pts)
        h = pts.squeeze(1)    # (B, 1024)
        h = self.fc1(h)
        h = self.fc2(h)
        return self.fc3(h).squeeze(-1)

print(f"PointNet++ params: {param_count(PointNetPP()):,}")

### 12.3 PointNext

In [ ]:
class InvResMLP(nn.Module):
    """Inverted Residual MLP block — core of PointNeXt."""

    def __init__(self, in_ch: int, expand: int = 4) -> None:
        super().__init__()
        mid = in_ch * expand
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, mid, 1),  nn.BatchNorm1d(mid),  nn.GELU(),
            nn.Conv1d(mid, in_ch, 1),  nn.BatchNorm1d(in_ch),
        )
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(x + self.block(x))


class PointNextEncoder(nn.Module):
    """
    PointNeXt-S style encoder.
    Stages: SA→IRM blocks→SA→IRM blocks → global pool.
    """

    def __init__(self, dropout: float = 0.4) -> None:
        super().__init__()
        # stem
        self.stem = nn.Sequential(
            nn.Conv1d(3, 64, 1), nn.BatchNorm1d(64), nn.GELU(),
        )
        # stage 1: SA + 2 IRM blocks
        self.sa1  = SetAbstraction(512, 0.2, 32, 0, [64, 128])
        self.irm1 = nn.Sequential(InvResMLP(128), InvResMLP(128))

        # stage 2: SA + 2 IRM blocks
        self.sa2  = SetAbstraction(128, 0.4, 64, 128, [128, 256])
        self.irm2 = nn.Sequential(InvResMLP(256), InvResMLP(256))

        # stage 3: global
        self.sa3  = GlobalSetAbstraction(256, [256, 512])

        self.fc1  = nn.Sequential(nn.Linear(512, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(dropout))
        self.fc2  = nn.Linear(256, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, 3, N)"""
        xyz = x.permute(0, 2, 1)   # (B, N, 3)

        xyz, pts = self.sa1(xyz, None)
        pts = self.irm1(pts.permute(0, 2, 1)).permute(0, 2, 1)

        xyz, pts = self.sa2(xyz, pts)
        pts = self.irm2(pts.permute(0, 2, 1)).permute(0, 2, 1)

        _, pts   = self.sa3(xyz, pts)
        h = pts.squeeze(1)

        h = self.fc1(h)
        return self.fc2(h).squeeze(-1)

# Alias for uniform interface
PointNext = PointNextEncoder

print(f"PointNext params: {param_count(PointNext()):,}")

### 12.4 PointMAE (masked autoencoder fine-tuned for regression)

In [ ]:
class PointPatchEmbed(nn.Module):
    """Divide point cloud into local patches, embed each with a mini-PointNet."""

    def __init__(self, num_groups: int = 64, group_size: int = 32, embed_dim: int = 256) -> None:
        super().__init__()
        self.num_groups  = num_groups
        self.group_size  = group_size
        self.embed_dim   = embed_dim

        self.encoder = nn.Sequential(
            nn.Conv1d(3, 64, 1),  nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64, 128, 1),nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, embed_dim, 1), nn.BatchNorm1d(embed_dim), nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        x: (B, 3, N)
        Returns:
          centers: (B, G, 3)
          tokens:  (B, G, D)
        """
        B = x.size(0)
        xyz = x.permute(0, 2, 1)   # (B, N, 3)

        # FPS for group centres
        fps_idx = _fps_torch(xyz, self.num_groups)      # (B, G)
        centers = _index_points(xyz, fps_idx)           # (B, G, 3)

        # kNN grouping around each centre
        dist = _square_distance(centers, xyz)           # (B, G, N)
        knn_idx = dist.topk(self.group_size, dim=-1, largest=False)[1]  # (B, G, k)
        grouped = _index_points(xyz, knn_idx.reshape(B, -1)).reshape(
            B, self.num_groups, self.group_size, 3
        )  # (B, G, k, 3)

        # normalise each patch
        grouped = grouped - centers.unsqueeze(2)

        # encode each patch with shared mini-PointNet
        g = grouped.reshape(B * self.num_groups, self.group_size, 3).permute(0, 2, 1)
        g = self.encoder(g).max(-1)[0]                        # (B*G, D)
        tokens = g.reshape(B, self.num_groups, self.embed_dim)
        return centers, tokens


class PointMAEBlock(nn.Module):
    """Standard pre-norm Transformer encoder block."""

    def __init__(self, dim: int = 256, num_heads: int = 4, mlp_ratio: float = 4.0, dropout: float = 0.1) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        mlp_dim    = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, mlp_dim), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.mlp(self.norm2(x))
        return x


class PointMAE(nn.Module):
    """
    PointMAE-style model (Point-MAE, Pang et al., 2022) fine-tuned for regression.
    The mask-decoder used for pre-training is omitted; we attach a regression head
    directly on top of the Transformer encoder (like the fine-tuning setup in the paper).
    """

    def __init__(
        self,
        num_groups:  int   = 64,
        group_size:  int   = 32,
        embed_dim:   int   = 256,
        depth:       int   = 6,
        num_heads:   int   = 4,
        dropout:     float = 0.3,
        mask_ratio:  float = POINTMAE_MASK_RATIO,
    ) -> None:
        super().__init__()
        self.mask_ratio  = mask_ratio
        self.embed_dim   = embed_dim
        self.patch_embed = PointPatchEmbed(num_groups, group_size, embed_dim)
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed   = nn.Sequential(
            nn.Linear(3, 128), nn.GELU(), nn.Linear(128, embed_dim)
        )
        self.blocks      = nn.ModuleList([
            PointMAEBlock(embed_dim, num_heads, dropout=dropout) for _ in range(depth)
        ])
        self.norm        = nn.LayerNorm(embed_dim)
        self.head        = nn.Sequential(
            nn.Linear(embed_dim, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, 3, N)"""
        centers, tokens = self.patch_embed(x)          # (B,G,3), (B,G,D)

        # positional embedding
        pos = self.pos_embed(centers)                  # (B, G, D)
        tokens = tokens + pos

        # prepend [CLS] token
        B = tokens.size(0)
        cls = self.cls_token.expand(B, -1, -1)        # (B, 1, D)
        tokens = torch.cat([cls, tokens], dim=1)       # (B, 1+G, D)

        # Transformer encoder
        for blk in self.blocks:
            tokens = blk(tokens)
        tokens = self.norm(tokens)

        # regression on [CLS] token
        cls_out = tokens[:, 0, :]                     # (B, D)
        return self.head(cls_out).squeeze(-1)

print(f"PointMAE params: {param_count(PointMAE()):,}")

### 12.5 DGCNN

In [ ]:
# ─── DGCNN helpers (reused & cleaned from original notebook) ───────────────────

def _knn_graph(x: torch.Tensor, k: int) -> torch.Tensor:
    """
    x: (B, C, N)
    Returns idx (B, N, k) — k-NN indices.
    """
    inner = -2 * torch.matmul(x.transpose(2, 1), x)           # (B, N, N)
    xx    = torch.sum(x ** 2, dim=1, keepdim=True)            # (B, 1, N)
    sq    = xx + inner + xx.transpose(2, 1)                    # (B, N, N)  pairwise sq-dist
    return (-sq).topk(k=k, dim=-1)[1]                         # (B, N, k)


def _edge_features(x: torch.Tensor, k: int, idx: Optional[torch.Tensor] = None) -> torch.Tensor:
    """
    Build edge features: [x_j - x_i, x_i] for each neighbour.
    Returns (B, 2C, N, k).
    """
    B, C, N = x.size()
    if idx is None:
        idx = _knn_graph(x, k)

    base    = torch.arange(B, device=x.device).view(-1, 1, 1) * N
    idx_flat = (idx + base).view(-1)

    x_t     = x.transpose(2, 1).contiguous()           # (B, N, C)
    nbr     = x_t.view(B * N, C)[idx_flat].view(B, N, k, C)   # (B, N, k, C)
    xi      = x_t.unsqueeze(2).expand_as(nbr)          # (B, N, k, C)
    edge    = torch.cat([nbr - xi, xi], dim=-1)        # (B, N, k, 2C)
    return edge.permute(0, 3, 2, 1).contiguous()       # (B, 2C, k, N)  ← Conv2d compatible


def _edge_block(in_ch: int, out_ch: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.LeakyReLU(negative_slope=0.2, inplace=True),
    )


class DGCNN(nn.Module):
    """
    DGCNN (Wang et al., 2019) — Dynamic Graph CNN for regression.
    Faithfully reproduces the 4-stage edge-conv + global descriptor design.
    """

    def __init__(self, k: int = DGCNN_K, dropout: float = 0.4) -> None:
        super().__init__()
        self.k = k

        self.ec1 = _edge_block(3  * 2, 64)
        self.ec2 = _edge_block(64 * 2, 64)
        self.ec3 = _edge_block(64 * 2, 128)
        self.ec4 = _edge_block(128* 2, 256)

        # aggregate across neighbour dimension  (B, C, 1, N) → (B, C, N)
        self.agg = lambda feat: feat.max(dim=2)[0]

        self.conv5  = nn.Sequential(
            nn.Conv1d(64 + 64 + 128 + 256, 1024, 1, bias=False),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
        )

        # global descriptor: cat(max, avg)
        self.fc1 = nn.Sequential(nn.Linear(2048, 512, bias=False), nn.BatchNorm1d(512),
                                  nn.LeakyReLU(0.2, inplace=True), nn.Dropout(dropout))
        self.fc2 = nn.Sequential(nn.Linear(512, 256), nn.BatchNorm1d(256),
                                  nn.LeakyReLU(0.2, inplace=True), nn.Dropout(dropout))
        self.fc3 = nn.Linear(256, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, 3, N)"""
        x1 = self.agg(self.ec1(_edge_features(x,  self.k)))   # (B, 64,  N)
        x2 = self.agg(self.ec2(_edge_features(x1, self.k)))   # (B, 64,  N)
        x3 = self.agg(self.ec3(_edge_features(x2, self.k)))   # (B, 128, N)
        x4 = self.agg(self.ec4(_edge_features(x3, self.k)))   # (B, 256, N)

        cat = torch.cat([x1, x2, x3, x4], dim=1)              # (B, 512, N)
        h   = self.conv5(cat)                                  # (B, 1024, N)

        h_max = h.max(dim=-1)[0]
        h_avg = h.mean(dim=-1)
        h     = torch.cat([h_max, h_avg], dim=1)              # (B, 2048)

        h = self.fc1(h)
        h = self.fc2(h)
        return self.fc3(h).squeeze(-1)

print(f"DGCNN params: {param_count(DGCNN()):,}")

## 13. Model Registry — Build the Selected Model

In [ ]:
MODEL_REGISTRY: Dict[str, Any] = {
    "PointNet":   PointNet,
    "PointNetPP": PointNetPP,
    "PointNext":  PointNext,
    "PointMAE":   PointMAE,
    "DGCNN":      DGCNN,
}

def build_model(name: str) -> nn.Module:
    if name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model '{name}'. Choose from {list(MODEL_REGISTRY)}")
    model = MODEL_REGISTRY[name]()
    n_params = param_count(model)
    print(f"Built  →  {name}   ({n_params:,} trainable params)")
    return model

model = build_model(MODEL_NAME)

## 14. Model Training

In [ ]:
set_seed(RANDOM_SEED)

trainer = Trainer(
    model        = model,
    model_name   = MODEL_NAME,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = DEVICE,
    lr           = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    grad_clip    = GRAD_CLIP,
    num_epochs   = NUM_EPOCHS,
    patience     = EARLY_STOPPING_PATIENCE,
    use_amp      = USE_AMP,
    ckpt_dir     = CHECKPOINTS_DIR,
    logs_dir     = LOGS_DIR,
)

print(f"\n{'='*60}")
print(f"  Training  :  {MODEL_NAME}")
print(f"  Epochs    :  {NUM_EPOCHS}  (patience={EARLY_STOPPING_PATIENCE})")
print(f"  Device    :  {DEVICE}")
print(f"{'='*60}\n")

best_val_metrics = trainer.train(resume=False)

### Plot Training History

In [ ]:
plot_training_history(trainer.history, MODEL_NAME)

## 15. Model Evaluation on Test Set

In [ ]:
# Re-build a clean model (no DataParallel wrapper) for the evaluator
eval_model = build_model(MODEL_NAME)

evaluator = Evaluator(
    model       = eval_model,
    model_name  = MODEL_NAME,
    test_loader = test_loader,
    device      = DEVICE,
    ckpt_dir    = CHECKPOINTS_DIR,
    plots_dir   = PLOTS_DIR,
)

test_metrics = evaluator.evaluate()
test_metrics["Model"]            = MODEL_NAME
test_metrics["Num_Params"]       = param_count(eval_model)
test_metrics["Train_Time_min"]   = round(trainer.total_train_time / 60, 2)

## 16. Inference on New Samples

In [ ]:
def infer_single(las_path: str, model: nn.Module, device: torch.device = DEVICE) -> float:
    """
    Load one .las file and predict volume using the given model.
    The model must already have weights loaded.
    """
    try:
        las = laspy.read(las_path)
        pts = np.vstack((las.x, las.y, las.z)).T.astype(np.float32)
    except Exception as exc:
        raise RuntimeError(f"Cannot read {las_path}: {exc}")

    pts = pts[np.isfinite(pts).all(axis=1)]
    if len(pts) == 0:
        raise ValueError(f"Empty point cloud: {las_path}")

    # FPS / resample
    ds = PointCloudDataset.__new__(PointCloudDataset)
    ds.num_points = NUM_POINTS
    pts = ds._sample(pts)
    pts = PointCloudDataset._normalise(pts)

    t = torch.from_numpy(pts.T).float().unsqueeze(0).to(device)   # (1, 3, N)

    model.eval()
    with torch.no_grad():
        out = model(t)
        if out.dim() == 0:
            out = out.unsqueeze(0)
    return float(out.squeeze().cpu())


# ── Example (run if you have a sample file) ───────────────────────────────────
sample_files = list(Path(DATA_DIR).glob("*.las"))
if sample_files:
    sample_path = str(sample_files[0])
    pred_vol = infer_single(sample_path, eval_model)
    print(f"File      : {sample_files[0].name}")
    print(f"Predicted Volume: {pred_vol:.4f}")
else:
    print("No .las files found in DATA_DIR for demo inference.")

## 17. Model Comparison

This cell reads all per-model test-metric CSVs and builds the comparison table. Run it after training every model.

In [ ]:
COMPARISON_CSV = os.path.join(RESULTS_DIR, "model_comparison.csv")

def update_comparison(metrics: Dict, csv_path: str = COMPARISON_CSV) -> pd.DataFrame:
    """
    Append or update a row in the comparison CSV and return the full table.
    """
    required = ["Model", "MAE", "RMSE", "MSE", "R2", "MAPE",
                "Inference_Time_s", "Num_Params", "Train_Time_min"]
    row_df = pd.DataFrame([{k: metrics.get(k, float("nan")) for k in required}])

    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        existing = existing[existing["Model"] != metrics["Model"]]  # remove old row
        combined = pd.concat([existing, row_df], ignore_index=True)
    else:
        combined = row_df

    combined = combined.sort_values("MAE").reset_index(drop=True)
    combined.to_csv(csv_path, index=False)
    return combined

comparison_df = update_comparison(test_metrics)

print("\n" + "=" * 90)
print("  OVERALL MODEL COMPARISON")
print("=" * 90)
print(comparison_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("=" * 90)
print(f"Saved → {COMPARISON_CSV}")

### Comparison Bar Charts (generated after all models are trained)

In [ ]:
def plot_comparison(csv_path: str = COMPARISON_CSV, plots_dir: str = PLOTS_DIR) -> None:
    if not os.path.exists(csv_path):
        print("Run all models first to generate the comparison CSV.")
        return
    df = pd.read_csv(csv_path)
    metrics = ["MAE", "RMSE", "MSE", "R2", "MAPE"]

    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 4))
    for ax, m in zip(axes, metrics):
        ax.bar(df["Model"], df[m], color=plt.cm.tab10.colors[:len(df)])
        ax.set_title(m)
        ax.set_xticklabels(df["Model"], rotation=25, ha="right")
    fig.suptitle("Model Comparison on Test Set", fontsize=14)
    fig.tight_layout()
    path = os.path.join(plots_dir, "model_comparison_bars.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Comparison chart saved → {path}")

plot_comparison()